In [7]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os
import time
import json

import torch
import torch.nn as nn
import torchvision.transforms as transforms

from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image

print("PyTorch version:", torch.__version__)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cpu
Device: cpu


In [9]:
DATASET_ROOT = "/content/drive/MyDrive/Dataset/plantwild"

DRIVE_MODEL_DIR = "/content/drive/MyDrive/PlantWild_Models"

os.makedirs(
    DRIVE_MODEL_DIR,
    exist_ok=True
)

print("Dataset path:")
print(DATASET_ROOT)

print("\nModel storage:")
print(DRIVE_MODEL_DIR)

Dataset path:
/content/drive/MyDrive/Dataset/plantwild

Model storage:
/content/drive/MyDrive/PlantWild_Models


In [10]:
trainval_file = os.path.join(
    DATASET_ROOT,
    "trainval.txt"
)

classes_file = os.path.join(
    DATASET_ROOT,
    "classes.txt"
)

images_folder = os.path.join(
    DATASET_ROOT,
    "images"
)

print(
    "Dataset exists:",
    os.path.exists(DATASET_ROOT)
)

print(
    "trainval.txt exists:",
    os.path.exists(trainval_file)
)

print(
    "classes.txt exists:",
    os.path.exists(classes_file)
)

print(
    "images folder exists:",
    os.path.exists(images_folder)
)

Dataset exists: True
trainval.txt exists: True
classes.txt exists: True
images folder exists: True


In [11]:
from collections import Counter

split_counts = Counter()

with open(
    trainval_file,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        image_path, class_id, mode = line.rsplit("=", 2)

        split_counts[int(mode)] += 1


print("Dataset split counts:\n")

for mode, count in sorted(split_counts.items()):

    print(
        f"Mode {mode}: {count} images"
    )

Dataset split counts:

Mode 0: 3677 images
Mode 1: 13045 images
Mode 2: 1820 images


In [12]:
CLASS_NAMES = []

with open(
    classes_file,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        class_id, class_name = line.split(
            " ",
            1
        )

        CLASS_NAMES.append(
            class_name
        )


NUM_CLASSES = len(CLASS_NAMES)

print(
    f"Number of classes: {NUM_CLASSES}"
)

print("\nFirst 10 classes:")

for i, name in enumerate(
    CLASS_NAMES[:10]
):

    print(
        i,
        "→",
        name
    )

Number of classes: 89

First 10 classes:
0 → apple black rot
1 → apple leaf
2 → apple mosaic virus
3 → apple rust
4 → apple scab
5 → banana leaf
6 → banana panama disease
7 → basil downy mildew
8 → basil leaf
9 → bean halo blight


In [13]:
transform_pipeline = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [14]:
class PlantWildDataset(Dataset):

    def __init__(self, split_mode):

        self.samples = []

        with open(
            trainval_file,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                image_path, class_id, mode = line.rsplit(
                    "=",
                    2
                )

                if int(mode) == split_mode:

                    full_path = os.path.join(
                        images_folder,
                        image_path
                    )

                    self.samples.append(
                        (
                            full_path,
                            int(class_id)
                        )
                    )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, index):

        image_path, label = self.samples[index]

        image = Image.open(
            image_path
        ).convert("RGB")

        image = transform_pipeline(
            image
        )

        return image, label

In [15]:
train_dataset = PlantWildDataset(
    split_mode=1
)

val_dataset = PlantWildDataset(
    split_mode=2
)

print(
    "Training images:",
    len(train_dataset)
)

print(
    "Validation images:",
    len(val_dataset)
)

Training images: 13045
Validation images: 1820


In [10]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training DataLoader ready.")
print("Validation DataLoader ready.")

Training DataLoader ready.
Validation DataLoader ready.


In [16]:
def create_mobilenet_model(
    num_classes=NUM_CLASSES
):

    model = models.mobilenet_v3_large(
        weights=models.MobileNet_V3_Large_Weights.DEFAULT
    )

    in_features = model.classifier[3].in_features

    model.classifier[3] = nn.Sequential(

        nn.Linear(
            in_features,
            512
        ),

        nn.Hardswish(),

        nn.Dropout(
            p=0.2
        ),

        nn.Linear(
            512,
            num_classes
        )
    )

    return model


model = create_mobilenet_model().to(device)

print(
    f"MobileNetV3-Large initialized."
)

print(
    f"Output classes: {NUM_CLASSES}"
)

print(
    f"Device: {device}"
)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 171MB/s]

MobileNetV3-Large initialized.
Output classes: 89
Device: cpu


In [17]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

print(
    "Loss:",
    criterion
)

print(
    "Optimizer:",
    optimizer.__class__.__name__
)

print(
    "Scheduler:",
    scheduler.__class__.__name__
)

Loss: CrossEntropyLoss()
Optimizer: AdamW
Scheduler: ReduceLROnPlateau


In [18]:
NUM_EPOCHS = 5

best_val_accuracy = 0.0

training_history = []

best_model_path = os.path.join(
    DRIVE_MODEL_DIR,
    "best_model.pth"
)

history_path = os.path.join(
    DRIVE_MODEL_DIR,
    "training_history.json"
)

print(
    "Number of epochs:",
    NUM_EPOCHS
)

print(
    "Best model will be saved to:"
)

print(
    best_model_path
)

Number of epochs: 5
Best model will be saved to:
/content/drive/MyDrive/PlantWild_Models/best_model.pth


In [16]:
print("\n" + "=" * 70)
print("STARTING PLANTWILD TRAINING")
print("=" * 70)

print(
    f"Training images:   {len(train_dataset)}"
)

print(
    f"Validation images: {len(val_dataset)}"
)

print(
    "Batch size:        32"
)

print(
    f"Epochs:            {NUM_EPOCHS}"
)

print(
    f"Device:            {device}"
)


for epoch in range(NUM_EPOCHS):

    epoch_start = time.time()

    # ==========================================================
    # TRAINING
    # ==========================================================

    model.train()

    running_loss = 0.0
    total_correct = 0
    total_images = 0
    total_trained = 0
    next_train_report = 100

    print("\n" + "-" * 70)
    print(
        f"EPOCH {epoch + 1}/{NUM_EPOCHS} - TRAINING"
    )
    print("-" * 70)

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        batch_size_actual = labels.size(0)

        running_loss += (
            loss.item()
            * batch_size_actual
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        total_correct += (
            predictions == labels
        ).sum().item()

        total_images += batch_size_actual

        total_trained += batch_size_actual

        while total_trained >= next_train_report:

            elapsed = (
                time.time()
                - epoch_start
            )

            current_loss = (
                running_loss
                / total_images
            )

            current_accuracy = (
                total_correct
                / total_images
            )

            print(
                f"✅ {next_train_report} "
                f"images trained "
                f"— Loss: {current_loss:.4f} "
                f"— Accuracy: {current_accuracy:.4f} "
                f"— {elapsed:.2f}s"
            )

            next_train_report += 100


    train_loss = (
        running_loss
        / total_images
    )

    train_accuracy = (
        total_correct
        / total_images
    )


    # ==========================================================
    # VALIDATION
    # ==========================================================

    print("\n🔎 Starting validation...")

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    total_validated = 0
    next_val_report = 100

    validation_start = time.time()

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            batch_size_actual = labels.size(0)

            val_loss += (
                loss.item()
                * batch_size_actual
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += batch_size_actual

            total_validated += batch_size_actual

            while total_validated >= next_val_report:

                print(
                    f"🔎 {next_val_report} "
                    f"validation images evaluated"
                )

                next_val_report += 100


    val_loss = (
        val_loss
        / val_total
    )

    val_accuracy = (
        val_correct
        / val_total
    )

    validation_time = (
        time.time()
        - validation_start
    )

    epoch_time = (
        time.time()
        - epoch_start
    )


    # ==========================================================
    # LEARNING RATE
    # ==========================================================

    scheduler.step(
        val_accuracy
    )

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )


    # ==========================================================
    # RESULTS
    # ==========================================================

    print("\n" + "=" * 70)
    print(
        f"EPOCH {epoch + 1} RESULTS"
    )
    print("=" * 70)

    print(
        f"Training Loss:       {train_loss:.4f}"
    )

    print(
        f"Training Accuracy:   {train_accuracy:.4f}"
    )

    print(
        f"Validation Loss:     {val_loss:.4f}"
    )

    print(
        f"Validation Accuracy: {val_accuracy:.4f}"
    )

    print(
        f"Learning Rate:       {current_lr:.8f}"
    )

    print(
        f"Validation Time:     {validation_time:.2f} seconds"
    )

    print(
        f"Total Epoch Time:    {epoch_time:.2f} seconds"
    )


    # ==========================================================
    # SAVE HISTORY
    # ==========================================================

    epoch_record = {

        "epoch": epoch + 1,

        "train_loss": train_loss,

        "train_accuracy": train_accuracy,

        "val_loss": val_loss,

        "val_accuracy": val_accuracy,

        "learning_rate": current_lr,

        "epoch_time": epoch_time
    }

    training_history.append(
        epoch_record
    )


    with open(
        history_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            training_history,
            f,
            indent=4
        )


    # ==========================================================
    # SAVE BEST MODEL
    # ==========================================================

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print("\n🏆 NEW BEST MODEL!")

        print(
            f"Best Validation Accuracy: "
            f"{best_val_accuracy:.4f}"
        )

        print(
            "Saved to:"
        )

        print(
            best_model_path
        )

    else:

        print(
            "\nBest Validation Accuracy remains: "
            f"{best_val_accuracy:.4f}"
        )


# ==============================================================
# TRAINING COMPLETE
# ==============================================================

print("\n" + "=" * 70)
print("🎉 TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best Validation Accuracy: "
    f"{best_val_accuracy:.4f}"
)

print("\nBest model:")

print(
    best_model_path
)

print("\nTraining history:")

print(
    history_path
)


STARTING PLANTWILD TRAINING
Training images:   13045
Validation images: 1820
Batch size:        32
Epochs:            5
Device:            cuda

----------------------------------------------------------------------
EPOCH 1/5 - TRAINING
----------------------------------------------------------------------
✅ 100 images trained — Loss: 0.5736 — Accuracy: 0.8359 — 3.79s
✅ 200 images trained — Loss: 0.5505 — Accuracy: 0.8304 — 5.68s
✅ 300 images trained — Loss: 0.5119 — Accuracy: 0.8469 — 6.54s
✅ 400 images trained — Loss: 0.5130 — Accuracy: 0.8365 — 8.03s
✅ 500 images trained — Loss: 0.5117 — Accuracy: 0.8359 — 9.20s
✅ 600 images trained — Loss: 0.5193 — Accuracy: 0.8339 — 10.59s
✅ 700 images trained — Loss: 0.5231 — Accuracy: 0.8352 — 11.39s
✅ 800 images trained — Loss: 0.5158 — Accuracy: 0.8387 — 12.50s
✅ 900 images trained — Loss: 0.5257 — Accuracy: 0.8297 — 14.55s
✅ 1000 images trained — Loss: 0.5326 — Accuracy: 0.8301 — 16.43s
✅ 1100 images trained — Loss: 0.5312 — Accuracy: 0.8277

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


✅ 12000 images trained — Loss: 0.5174 — Accuracy: 0.8371 — 172.56s
✅ 12100 images trained — Loss: 0.5168 — Accuracy: 0.8371 — 174.02s
✅ 12200 images trained — Loss: 0.5153 — Accuracy: 0.8376 — 174.96s
✅ 12300 images trained — Loss: 0.5163 — Accuracy: 0.8373 — 177.53s
✅ 12400 images trained — Loss: 0.5190 — Accuracy: 0.8366 — 178.70s
✅ 12500 images trained — Loss: 0.5188 — Accuracy: 0.8365 — 180.88s
✅ 12600 images trained — Loss: 0.5181 — Accuracy: 0.8365 — 181.60s
✅ 12700 images trained — Loss: 0.5176 — Accuracy: 0.8367 — 183.16s
✅ 12800 images trained — Loss: 0.5187 — Accuracy: 0.8363 — 184.19s
✅ 12900 images trained — Loss: 0.5184 — Accuracy: 0.8362 — 185.86s
✅ 13000 images trained — Loss: 0.5180 — Accuracy: 0.8363 — 186.86s

🔎 Starting validation...
🔎 100 validation images evaluated
🔎 200 validation images evaluated
🔎 300 validation images evaluated
🔎 400 validation images evaluated
🔎 500 validation images evaluated
🔎 600 validation images evaluated
🔎 700 validation images evaluated

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


✅ 4900 images trained — Loss: 0.3842 — Accuracy: 0.8793 — 71.47s
✅ 5000 images trained — Loss: 0.3828 — Accuracy: 0.8798 — 73.40s
✅ 5100 images trained — Loss: 0.3845 — Accuracy: 0.8791 — 74.10s
✅ 5200 images trained — Loss: 0.3828 — Accuracy: 0.8796 — 75.36s
✅ 5300 images trained — Loss: 0.3841 — Accuracy: 0.8793 — 76.03s
✅ 5400 images trained — Loss: 0.3824 — Accuracy: 0.8800 — 77.64s
✅ 5500 images trained — Loss: 0.3841 — Accuracy: 0.8795 — 78.25s
✅ 5600 images trained — Loss: 0.3838 — Accuracy: 0.8795 — 79.54s
✅ 5700 images trained — Loss: 0.3860 — Accuracy: 0.8785 — 82.44s
✅ 5800 images trained — Loss: 0.3849 — Accuracy: 0.8789 — 85.10s
✅ 5900 images trained — Loss: 0.3853 — Accuracy: 0.8792 — 85.71s
✅ 6000 images trained — Loss: 0.3856 — Accuracy: 0.8792 — 86.92s
✅ 6100 images trained — Loss: 0.3855 — Accuracy: 0.8793 — 87.72s
✅ 6200 images trained — Loss: 0.3856 — Accuracy: 0.8787 — 89.13s
✅ 6300 images trained — Loss: 0.3864 — Accuracy: 0.8786 — 89.96s
✅ 6400 images trained — L

...

✅ 900 images trained — Loss: 0.2905 — Accuracy: 0.9019 — 13.57s
✅ 1000 images trained — Loss: 0.2869 — Accuracy: 0.9043 — 15.60s
✅ 1100 images trained — Loss: 0.2837 — Accuracy: 0.9071 — 16.58s
✅ 1200 images trained — Loss: 0.2902 — Accuracy: 0.9062 — 18.49s
✅ 1300 images trained — Loss: 0.2866 — Accuracy: 0.9093 — 20.09s
✅ 1400 images trained — Loss: 0.2818 — Accuracy: 0.9119 — 22.16s
✅ 1500 images trained — Loss: 0.2783 — Accuracy: 0.9142 — 22.90s
✅ 1600 images trained — Loss: 0.2837 — Accuracy: 0.9125 — 24.41s
✅ 1700 images trained — Loss: 0.2788 — Accuracy: 0.9155 — 25.83s
✅ 1800 images trained — Loss: 0.2732 — Accuracy: 0.9167 — 26.64s
✅ 1900 images trained — Loss: 0.2708 — Accuracy: 0.9182 — 28.21s
✅ 2000 images trained — Loss: 0.2743 — Accuracy: 0.9177 — 29.04s
✅ 2100 images trained — Loss: 0.2715 — Accuracy: 0.9181 — 30.22s
✅ 2200 images trained — Loss: 0.2700 — Accuracy: 0.9185 — 31.24s
✅ 2300 images trained — Loss: 0.2677 — Accuracy: 0.9188 — 34.22s
✅ 2400 images trained — Lo

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


✅ 3200 images trained — Loss: 0.1963 — Accuracy: 0.9391 — 45.83s
✅ 3300 images trained — Loss: 0.1975 — Accuracy: 0.9390 — 46.97s
✅ 3400 images trained — Loss: 0.1989 — Accuracy: 0.9390 — 47.95s
✅ 3500 images trained — Loss: 0.2007 — Accuracy: 0.9381 — 49.06s
✅ 3600 images trained — Loss: 0.2002 — Accuracy: 0.9383 — 49.99s
✅ 3700 images trained — Loss: 0.1994 — Accuracy: 0.9388 — 51.40s
✅ 3800 images trained — Loss: 0.2002 — Accuracy: 0.9393 — 53.06s
✅ 3900 images trained — Loss: 0.2008 — Accuracy: 0.9390 — 56.50s
✅ 4000 images trained — Loss: 0.2003 — Accuracy: 0.9390 — 57.36s
✅ 4100 images trained — Loss: 0.2010 — Accuracy: 0.9390 — 58.62s
✅ 4200 images trained — Loss: 0.2020 — Accuracy: 0.9387 — 59.82s
✅ 4300 images trained — Loss: 0.2046 — Accuracy: 0.9373 — 60.85s
✅ 4400 images trained — Loss: 0.2041 — Accuracy: 0.9373 — 62.01s
✅ 4500 images trained — Loss: 0.2033 — Accuracy: 0.9377 — 63.33s
✅ 4600 images trained — Loss: 0.2062 — Accuracy: 0.9368 — 65.26s
✅ 4700 images trained — L

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


✅ 12000 images trained — Loss: 0.1762 — Accuracy: 0.9468 — 173.77s
✅ 12100 images trained — Loss: 0.1761 — Accuracy: 0.9469 — 175.29s
✅ 12200 images trained — Loss: 0.1757 — Accuracy: 0.9470 — 176.75s
✅ 12300 images trained — Loss: 0.1755 — Accuracy: 0.9470 — 177.60s
✅ 12400 images trained — Loss: 0.1757 — Accuracy: 0.9469 — 178.88s
✅ 12500 images trained — Loss: 0.1757 — Accuracy: 0.9469 — 179.68s
✅ 12600 images trained — Loss: 0.1762 — Accuracy: 0.9466 — 182.06s
✅ 12700 images trained — Loss: 0.1762 — Accuracy: 0.9466 — 184.62s
✅ 12800 images trained — Loss: 0.1766 — Accuracy: 0.9464 — 186.52s
✅ 12900 images trained — Loss: 0.1778 — Accuracy: 0.9461 — 188.18s
✅ 13000 images trained — Loss: 0.1779 — Accuracy: 0.9463 — 188.87s

🔎 Starting validation...
🔎 100 validation images evaluated
🔎 200 validation images evaluated
🔎 300 validation images evaluated
🔎 400 validation images evaluated
🔎 500 validation images evaluated
🔎 600 validation images evaluated
🔎 700 validation images evaluated

In [17]:
print(
    "Best model exists:",
    os.path.exists(best_model_path)
)

if os.path.exists(best_model_path):

    size_mb = (
        os.path.getsize(best_model_path)
        / (1024 ** 2)
    )

    print(
        f"Model size: {size_mb:.2f} MB"
    )

print(
    "\nTraining history exists:",
    os.path.exists(history_path)
)

Best model exists: True
Model size: 18.90 MB

Training history exists: True


In [19]:
test_dataset = PlantWildDataset(
    split_mode=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(
    "Test images:",
    len(test_dataset)
)

Test images: 3677


In [20]:
model = create_mobilenet_model().to(device)

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

model.eval()

print(
    "✅ Best model loaded successfully."
)

✅ Best model loaded successfully.


In [23]:

import time
import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

test_start = time.time()

test_correct = 0
test_total = 0

all_predictions = []
all_labels = []

print("\nStarting final test evaluation...\n")

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_correct += (
            predictions == labels
        ).sum().item()

        test_total += labels.size(0)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        if test_total % 100 < 32:
            print(
                f"Evaluated {test_total} test images"
            )


# ==================================================
# METRICS
# ==================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

# -----------------------------
# MICRO METRICS
# -----------------------------

precision_micro = precision_score(
    all_labels,
    all_predictions,
    average="micro",
    zero_division=0
)

recall_micro = recall_score(
    all_labels,
    all_predictions,
    average="micro",
    zero_division=0
)

f1_micro = f1_score(
    all_labels,
    all_predictions,
    average="micro",
    zero_division=0
)

# -----------------------------
# MACRO METRICS
# -----------------------------

precision_macro = precision_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    all_labels,
    all_predictions,
    average="macro",
    zero_division=0
)

# -----------------------------
# WEIGHTED METRICS
# -----------------------------

precision_weighted = precision_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

recall_weighted = recall_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)

f1_weighted = f1_score(
    all_labels,
    all_predictions,
    average="weighted",
    zero_division=0
)


test_time = time.time() - test_start


# ==================================================
# FINAL RESULTS
# ==================================================

print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(f"Test images:        {test_total}")
print(f"Correct:            {test_correct}")
print(f"Incorrect:          {test_total - test_correct}")

print("\n--- Overall ---")

print(f"Accuracy:            {accuracy:.4f}")
print(f"Accuracy %:          {accuracy * 100:.2f}%")

print("\n--- Micro Average ---")

print(f"Micro Precision:     {precision_micro:.4f}")
print(f"Micro Recall:        {recall_micro:.4f}")
print(f"Micro F1:            {f1_micro:.4f}")

print("\n--- Macro Average ---")

print(f"Macro Precision:     {precision_macro:.4f}")
print(f"Macro Recall:        {recall_macro:.4f}")
print(f"Macro F1:            {f1_macro:.4f}")

print("\n--- Weighted Average ---")

print(f"Weighted Precision:  {precision_weighted:.4f}")
print(f"Weighted Recall:     {recall_weighted:.4f}")
print(f"Weighted F1:         {f1_weighted:.4f}")

print(f"\nTest time:           {test_time:.2f} seconds")

print("=" * 60)



Starting final test evaluation...

Evaluated 128 test images
Evaluated 224 test images
Evaluated 320 test images
Evaluated 416 test images
Evaluated 512 test images
Evaluated 608 test images
Evaluated 704 test images
Evaluated 800 test images
Evaluated 928 test images
Evaluated 1024 test images
Evaluated 1120 test images
Evaluated 1216 test images
Evaluated 1312 test images
Evaluated 1408 test images
Evaluated 1504 test images
Evaluated 1600 test images
Evaluated 1728 test images
Evaluated 1824 test images
Evaluated 1920 test images
Evaluated 2016 test images
Evaluated 2112 test images
Evaluated 2208 test images
Evaluated 2304 test images
Evaluated 2400 test images
Evaluated 2528 test images
Evaluated 2624 test images
Evaluated 2720 test images
Evaluated 2816 test images
Evaluated 2912 test images
Evaluated 3008 test images
Evaluated 3104 test images
Evaluated 3200 test images
Evaluated 3328 test images
Evaluated 3424 test images
Evaluated 3520 test images
Evaluated 3616 test images

